In [39]:
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3


doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [40]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [41]:
TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"

input_features = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 
                'B9',
               "B4_minus_B3",
               "B4_div_B3", 
               "ND_B4_B3", 
               'Temperature', 'DewPoint', 'v10n', 
               'doy_sin', 'doy_cos', 
               'latitude', 'longitude'
               ]

X_train = doc_train_augmented[input_features]
y_train = doc_train_augmented[TARGET]

X_test = doc_test[input_features]
y_test = doc_test[TARGET]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

#y_test

((2732, 19), (2732,), (171, 19), (171,))

In [42]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print(SVR.__module__)

sklearn.svm._classes


In [43]:
svr_base_WM = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",
        ))
])

svr_base_WM.fit(X_train, y_train)
y_pred = svr_base_WM.predict(X_test)
print("R² for DOC for log scaled | Baseline SVM Regressor:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | Baseline SVM Regressor:", r2_score(y_test_real, y_pred_orig))


# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



R² for DOC for log scaled | Baseline SVM Regressor: 0.43738098934622505
R² for DOC on normal scaled | Baseline SVM Regressor: 0.31449503704003734
R² for DOC on normal: 0.31449503704003734
Final metrics on NORMAL scale (after inverse log):
R²   : 0.3145
MSE  : 2.1377
RMSE : 1.4621
MAE  : 0.8924


In [ ]:
#Hyper Param tuning
from sklearn.model_selection import KFold
kf = KFold(n_splits=3, shuffle=True, random_state=42)

def objective_svr_WM(trial):

    params = {
        "C": trial.suggest_float("C", 1e-2, 1e3, log=True),
        "epsilon": trial.suggest_float("epsilon", 1e-3, 1.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 1.0, log=True),
        "kernel": "rbf"
    }

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(**params))
    ])

    r2_scores = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)

        r2_scores.append(r2_score(y_va, preds))

    return np.mean(r2_scores)

In [18]:
import optuna

study_svr_WM = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="SVR_withCV_WM"
)

study_svr_WM.optimize(objective_svr_WM, n_trials=50, show_progress_bar=True)

d:\Miniconda3\envs\PyTorchVirtualEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-02-09 10:40:53,164] A new study created in memory with name: SVR_withCV_WM
Best trial: 0. Best value: 0.183525:   0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-02-09 10:40:53,196] Trial 0 finished with value: 0.18352522453900852 and parameters: {'C': 0.7459343285726545, 'epsilon': 0.711447600934342, 'gamma': 0.08471801418819976}. Best is trial 0 with value: 0.18352522453900852.


Best trial: 2. Best value: 0.192438:   6%|▌         | 3/50 [00:00<00:09,  4.94it/s]

[I 2026-02-09 10:40:53,696] Trial 1 finished with value: 0.18258193677657708 and parameters: {'C': 9.846738873614559, 'epsilon': 0.0029380279387035343, 'gamma': 0.0004207053950287938}. Best is trial 0 with value: 0.18352522453900852.
[I 2026-02-09 10:40:53,811] Trial 2 finished with value: 0.19243842629458427 and parameters: {'C': 0.0195172246414495, 'epsilon': 0.39676050770529875, 'gamma': 0.02537815508265665}. Best is trial 2 with value: 0.19243842629458427.


Best trial: 3. Best value: 0.969651:   8%|▊         | 4/50 [00:01<00:19,  2.34it/s]

...

[I 2026-02-09 10:41:58,620] Trial 47 finished with value: 0.9754425231327474 and parameters: {'C': 5.1310600800916495, 'epsilon': 0.002481462266956343, 'gamma': 0.29605600958122313}. Best is trial 38 with value: 0.9759190794640077.


Best trial: 38. Best value: 0.975919:  98%|█████████▊| 49/50 [01:05<00:00,  1.43it/s]

[I 2026-02-09 10:41:59,115] Trial 48 finished with value: 0.13333285940637765 and parameters: {'C': 0.652617179551054, 'epsilon': 0.002514206766717188, 'gamma': 0.00019167808303473415}. Best is trial 38 with value: 0.9759190794640077.


Best trial: 38. Best value: 0.975919: 100%|██████████| 50/50 [01:06<00:00,  1.34s/it]

[I 2026-02-09 10:41:59,925] Trial 49 finished with value: 0.6295635295897036 and parameters: {'C': 7.071757800029679, 'epsilon': 0.004088948130935703, 'gamma': 0.025921297892045603}. Best is trial 38 with value: 0.9759190794640077.


In [44]:
print(study_svr_WM.best_params)
print(study_svr_WM.best_value)

{'C': 43.1558571666661, 'epsilon': 0.004752680680615191, 'gamma': 0.3096522957081244}
0.9759190794640077


In [45]:
best_params_WM = study_svr_WM.best_params
print(best_params_WM)

{'C': 43.1558571666661, 'epsilon': 0.004752680680615191, 'gamma': 0.3096522957081244}


In [46]:
best_params = best_params_WM

best_svr_WM = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(**best_params, kernel="rbf"))
])


best_svr_WM.fit(X_train, y_train)
y_pred = best_svr_WM.predict(X_test)

print("R² for DOC for log scaled | SVR:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | SVR:", r2_score(y_test_real, y_pred_orig))




# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

R² for DOC for log scaled | SVR: 0.381441609612687
R² for DOC on normal scaled | SVR: 0.3165146824425564
R² for DOC on normal: 0.3165146824425564
Final metrics on NORMAL scale (after inverse log):
R²   : 0.3165
MSE  : 2.1314
RMSE : 1.4599
MAE  : 0.9570


In [ ]:
#For without meteo

In [47]:
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3

doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [48]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [49]:
TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"

input_features_noMeteo = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 
                'B9',
               "B4_minus_B3",
               "B4_div_B3", 
               "ND_B4_B3", 
               
               #'Temperature', 'DewPoint', 'v10n', #'Precipitation(mm)',
               'doy_sin', 'doy_cos', 
               'latitude', 'longitude'
               ]

X_train = doc_train_augmented[input_features_noMeteo]
y_train = doc_train_augmented[TARGET]

X_test = doc_test[input_features_noMeteo]
y_test = doc_test[TARGET]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2732, 16), (2732,), (171, 16), (171,))

In [50]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print(SVR.__module__)

sklearn.svm._classes


In [51]:
svr_base_NM = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",
        ))
])

svr_base_NM.fit(X_train, y_train)
y_pred = svr_base_NM.predict(X_test)
print("R² for DOC for log scaled | Baseline SVM Regressor:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | Baseline SVM Regressor:", r2_score(y_test_real, y_pred_orig))


# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



R² for DOC for log scaled | Baseline SVM Regressor: 0.4635169797119849
R² for DOC on normal scaled | Baseline SVM Regressor: 0.34808861958344806
R² for DOC on normal: 0.34808861958344806
Final metrics on NORMAL scale (after inverse log):
R²   : 0.3481
MSE  : 2.0330
RMSE : 1.4258
MAE  : 0.8240


In [32]:
#Hyper Param tuning
from sklearn.model_selection import KFold
kf = KFold(n_splits=3, shuffle=True, random_state=42)

def objective_svr_NM(trial):

    params = {
        "C": trial.suggest_float("C", 1e-2, 1e3, log=True),
        "epsilon": trial.suggest_float("epsilon", 1e-3, 1.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 1.0, log=True),
        "kernel": "rbf"
    }

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(**params))
    ])

    r2_scores = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)

        r2_scores.append(r2_score(y_va, preds))

    return np.mean(r2_scores)

In [33]:
import optuna

study_svr_NM = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="SVR_withCV_WM"
)

study_svr_NM.optimize(objective_svr_NM, n_trials=50, show_progress_bar=True)

[I 2026-02-09 10:50:08,773] A new study created in memory with name: SVR_withCV_WM
Best trial: 0. Best value: 0.198228:   0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-02-09 10:50:08,805] Trial 0 finished with value: 0.19822753997936196 and parameters: {'C': 0.7459343285726545, 'epsilon': 0.711447600934342, 'gamma': 0.08471801418819976}. Best is trial 0 with value: 0.19822753997936196.


Best trial: 0. Best value: 0.198228:   6%|▌         | 3/50 [00:00<00:09,  5.01it/s]

[I 2026-02-09 10:50:09,291] Trial 1 finished with value: 0.17975294733073977 and parameters: {'C': 9.846738873614559, 'epsilon': 0.0029380279387035343, 'gamma': 0.0004207053950287938}. Best is trial 0 with value: 0.19822753997936196.
[I 2026-02-09 10:50:09,407] Trial 2 finished with value: 0.19417634995401736 and parameters: {'C': 0.0195172246414495, 'epsilon': 0.39676050770529875, 'gamma': 0.02537815508265665}. Best is trial 0 with value: 0.19822753997936196.


Best trial: 3. Best value: 0.965508:   8%|▊         | 4/50 [00:02<00:32,  1.40it/s]

...

[I 2026-02-09 10:51:54,168] Trial 46 finished with value: 0.4764824448329601 and parameters: {'C': 24.929074358345968, 'epsilon': 0.0025601546674484115, 'gamma': 0.011656244505437714}. Best is trial 45 with value: 0.9657888200081398.


Best trial: 45. Best value: 0.965789:  96%|█████████▌| 48/50 [01:46<00:05,  2.87s/it]

[I 2026-02-09 10:51:54,816] Trial 47 finished with value: 0.9639587285949878 and parameters: {'C': 3.104558008146649, 'epsilon': 0.00859464620848121, 'gamma': 0.6504646161935417}. Best is trial 45 with value: 0.9657888200081398.


Best trial: 45. Best value: 0.965789: 100%|██████████| 50/50 [01:46<00:00,  2.13s/it]

[I 2026-02-09 10:51:55,290] Trial 48 finished with value: 0.16667188915910988 and parameters: {'C': 8.442626019932026, 'epsilon': 0.0055557620637177695, 'gamma': 0.00019167808303473415}. Best is trial 45 with value: 0.9657888200081398.
[I 2026-02-09 10:51:55,380] Trial 49 finished with value: 0.4560334437549198 and parameters: {'C': 27.801964268371755, 'epsilon': 0.466304126169473, 'gamma': 0.10868303341793496}. Best is trial 45 with value: 0.9657888200081398.


In [52]:
print(study_svr_NM.best_params)
print(study_svr_NM.best_value)


{'C': 25.581263740035503, 'epsilon': 0.002592363878590811, 'gamma': 0.6898899913178642}
0.9657888200081398


In [53]:
best_params_NM = study_svr_NM.best_params
print(best_params_NM)

{'C': 25.581263740035503, 'epsilon': 0.002592363878590811, 'gamma': 0.6898899913178642}


In [ ]:
best_params = best_params_NM

best_svr_NM = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(**best_params, kernel="rbf"))
])


best_svr_NM.fit(X_train, y_train)
y_pred = best_svr_NM.predict(X_test)

print("R² for DOC for log scaled | SVR no Meteo:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | SVR No meteo:", r2_score(y_test_real, y_pred_orig))



# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))


# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

R² for DOC for log scaled | SVR no Meteo: 0.41259190421362124
R² for DOC on normal scaled | SVR No meteo: 0.37804843050474335
R² for DOC on normal: 0.37804843050474335
Final metrics on NORMAL scale (after inverse log):
R²   : 0.3780
MSE  : 1.9395
RMSE : 1.3927
MAE  : 0.9363


In [ ]:
#Completed